# 03 · Resultados: tablas GOLD y modelo base
Lee las tablas que el flujo de Prefect cargó en la base de datos (AWS RDS o SQLite local) y resume los hallazgos para el documento y la presentación.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
from src.utils import load_config

cfg = load_config()
FIG = cfg["paths"]["reports_dir"] / "figures"; FIG.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.float_format", "{:,.3f}".format)

def guardar(nombre):
    plt.tight_layout(); plt.savefig(FIG / f"{nombre}.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
from sqlalchemy import create_engine, inspect
from src.utils import get_db_url
engine = create_engine(get_db_url(cfg))
tablas = inspect(engine).get_table_names()
print("Tablas en la base de datos:", tablas)
leer = lambda t: pd.read_sql(f"SELECT * FROM {t}", engine)
leer("kpis").T

In [ ]:
cat = leer("by_category").head(15).sort_values("pct_negative")
ax = cat.plot.barh(x="main_category", y="pct_negative", figsize=(8, 6), color="#C44E52", legend=False)
ax.set_title("% de reseñas negativas por categoría (top 15 por volumen)"); ax.set_xlabel("% negativas")
guardar("11_negativas_por_categoria")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, (t, col) in zip(ax, [("by_text_length", "text_len_bucket"), ("by_user_activity", "user_activity"),
                            ("by_book_age", "book_age_bucket")]):
    d = leer(t).sort_values("orden"); a.bar(d[col], d["pct_negative"], color="#4C72B0")
    a.set_title(t); a.tick_params(axis="x", rotation=30); a.set_ylabel("% negativas")
guardar("12_segmentos")

In [ ]:
display(leer("model_metrics"))
imp = leer("model_feature_importance").head(12).sort_values("importance")
ax = imp.plot.barh(x="feature", y="importance", xerr="std", figsize=(8, 5), legend=False, color="#8172B2")
ax.set_title("Importancia por permutación (caída en ROC AUC)")
guardar("13_importancia_variables")

In [ ]:
leer("etl_runs")